# **02 웹 검색 에이전트 구현하기**

### 학습 내용
1. Tavily Search API 이해 및 설정
2. TavilySearch 도구 사용법
3. 웹 검색 결과 처리
4. create_agent와 Tavily Search 통합
5. 다중 도구 에이전트 구성

## 1. 환경 설정

- OpenAI API Key 발급: https://platform.openai.com/api-keys
- Tavily API Key 발급: https://app.tavily.com/home

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI API Key가 설정되었습니다.")

if os.getenv("TAVILY_API_KEY"):
    print("TAVILY API Key가 설정되었습니다.")

OPENAI API Key가 설정되었습니다.
TAVILY API Key가 설정되었습니다.


## 2. Tavily Search API 소개

**Tavily Search API**는 AI 에이전트(LLM)를 위해 특별히 설계된 검색 엔진입니다.

### 주요 특징

- **실시간 검색**: 최신 정보를 빠르게 검색
- **AI 최적화**: LLM이 이해하기 쉬운 형식으로 결과 제공
- **정확한 결과**: 신뢰할 수 있는 소스에서 팩트 기반 정보 제공
- **무료 티어**: 월 1,000회 무료 검색 제공

## 3. TavilySearch 도구 기본 사용법

[TavilySearch](https://reference.langchain.com/python/langchain-tavily/tavily_search/TavilySearch) 도구를 직접 사용하여 웹 검색을 수행해봅시다.

In [2]:
from langchain_tavily import TavilySearch

# 기본 설정으로 TavilySearch 도구 생성
search_tool = TavilySearch(
    max_results=3,  # 최대 검색 결과 수
    topic="general",  # 검색 카테고리: "general", "news", "finance"
)

print(f"도구 이름: {search_tool.name}")
print(f"도구 설명: {search_tool.description}")

도구 이름: tavily_search
도구 설명: A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. It not only retrieves URLs and snippets, but offers advanced search depths, domain management, time range filters, and image search, this tool delivers real-time, accurate, and citation-backed results.Input should be a search query.


In [3]:
result = search_tool.invoke({"query": "LangChain이란 무엇인가요?"})

print("\n=== 검색 결과 ===")
print(result)


=== 검색 결과 ===
{'query': 'LangChain이란 무엇인가요?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://cloud.google.com/use-cases/langchain?hl=ko', 'title': 'LangChain이란 무엇인가요? 예시 및 정의 - Google Cloud', 'content': 'LangChain은 대규모 언어 모델(LLM)로 애플리케이션을 더 쉽게 빌드할 수 있도록 지원하는 오픈소스 조정 프레임워크입니다. LLM을 다양한 데이터 소스에 연결하는 도구와 구성요소를 제공하여 복잡한 다단계 워크플로를 만들 수 있습니다. LangChain은 Python 및 JavaScript 라이브러리로 제공되며, 개발자가 LLM을 외부 데이터 및 계산에 연결하여 텍스트 생성 이상의 LLM 기능을 향상하도록 지원합니다. 이를 통해 지능형 챗봇, 정교한 질의 응답 시스템, 자동화된 데이터 분석 도구와 같은 고급 AI 애플리케이션의 개발을 촉진할 수 있습니다. 사용자가 쿼리를 제출하면 LangChain은 일련의 단계를 거쳐 이 입력을 처리할 수 있습니다. LangChain은 이러한 단계를 구성요소로 간소화하여 LLM 또는 외부 리소스와 여러 번 상호작용해야 하는 애플리케이션을 더 쉽게 빌드할 수 있도록 해 줍니다. 또한 이 프레임워크는 다양한 LLM을 사용하는 방법을 제공하여 개발자가 특정 애플리케이션에 가장 적합한 모델을 자유롭게 선택할 수 있도록 지원합니다. LLM을 메모리 및 외부 지식과 통합하여 컨텍스트를 유지하고, 질문에 답하고, 자연어 대화에 참여할 수 있는 정교한 챗봇을 빌드합니다. 정형 또는 비정형 데이터 소스와 상호작용하여 자연어 쿼리를 기반으로 정보를 검색, 분석, 요약할 수 있는 애플리케이션을 빌드합니다.', 'score': 0.9317675, 'raw_content': None}, {'url': 'https

### 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|-------|
| `max_results` | 반환할 최대 검색 결과 수 | 5 |
| `topic` | 검색 카테고리 ("general", "news", "finance") | "general" |
| `include_answer` | 질문에 대한 직접 답변 포함 여부 | False |
| `include_raw_content` | 전체 HTML 콘텐츠 포함 여부 | False |
| `include_images` | 관련 이미지 포함 여부 | False |
| `search_depth` | 검색 깊이 ("basic", "advanced") | "basic" |
| `include_domains` | 포함할 도메인 리스트 | None |
| `exclude_domains` | 제외할 도메인 리스트 | None |
| `time_range` | 시간 범위 ("day", "week", "month", "year") | None |

## 4. 특정 도메인으로 검색 제한하기

특정 웹사이트만 검색하거나 특정 사이트를 제외할 수 있습니다.

In [4]:
# Wikipedia만 검색하기
wiki_search = TavilySearch(
    max_results=3,
    include_domains=["wikipedia.org"],  # Wikipedia만 검색
)

result = wiki_search.invoke({"query": "인공지능"})

print("\n=== Wikipedia 검색 결과 ===")
print(result)


=== Wikipedia 검색 결과 ===
{'query': '인공지능', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://ko.wikipedia.org/wiki/%EC%9D%B8%EA%B3%B5_%EC%9D%BC%EB%B0%98_%EC%A7%80%EB%8A%A5', 'title': '인공 일반 지능 - 위키백과, 우리 모두의 백과사전', 'content': '**인공 일반 지능**(人工一般智能, 영어: artificial general intelligence, **AGI**)은 사실상 모든 인지 작업에서 인간의 능력과 같거나 능가할 수 있는 인공지능의 한 유형이다. AI 선구자 허버트 사이먼은 1965년에 "기계는 20년 안에 인간이 할 수 있는 어떤 작업이든 할 수 있을 것"이라고 추측했다. * **강한 AI 가설**: 인공지능 시스템은 "마음"과 "의식"을 가질 수 있다. ↑ Wiggers, Kyle (2022년 5월 13일), “DeepMind\'s new AI can perform over 600 tasks, from playing games to controlling robots”, 《테크크런치》, 2022년 6월 16일에 원본 문서에서 보존된 문서, 2022년 6월 12일에 확인함. 《A Review of Artificial Intelligence (AI) in Education from 2010 to 2020》 (영어). ↑ “Stephen Hawking: \'Transcendence looks at the implications of artificial intelligence;– but are we taking AI seriously enough?\'”. * Goertzel, Ben") (Dec 2007), “Human-level artificial general intelligence and the possibility of a te

In [5]:
# 특정 도메인 제외하기
filtered_search = TavilySearch(
    max_results=3,
    exclude_domains=["reddit.com", "twitter.com", "blog.naver.com", "tistory.com"],
)

result = filtered_search.invoke({"query": "파이썬 프로그래밍 팁"})

print("\n=== 필터링된 검색 결과 ===")
print(result)


=== 필터링된 검색 결과 ===
{'query': '파이썬 프로그래밍 팁', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://kr.linkedin.com/pulse/10-tips-ideas-python-beginners-expert-journey-ayush-gupta-pfljf?tl=ko', 'title': '파이썬 초보자부터 전문가까지 여정을 위한 10가지 팁과 아이디어', 'content': '웹 개발을 위해 파이썬을 배우고 싶다면 Django나 Flask 같은 프레임워크에 집중하는 게 좋을 것 같아요. 목표를 알면 자신의 필요와 관심사에 맞게 학습', 'score': 0.9979652, 'raw_content': None}, {'url': 'https://medium.com/@easydatascience/python-%ED%8C%8C%EC%9D%B4%EC%8D%AC-%EC%96%B4%EB%96%BB%EA%B2%8C-%EC%8B%9C%EC%9E%91%ED%95%B4%EC%95%BC-%ED%95%A0%EA%B9%8C-%ED%8C%8C%EC%9D%B4%EC%8D%AC-%EC%8B%9C%EC%9E%91-tip-5%EA%B0%80%EC%A7%80-f1e2ed0808bb', 'title': 'Python(파이썬) 어떻게 시작해야 할까? — 파이썬 시작 Tip 5가지', 'content': '파이썬 시작 Tip · 일단 시작하는게 제일 중요합니다. 많은 사람들이 코딩을 하는게 도움될까,말까 주저합니다. · 파이썬 설치는 아주 쉽습니다. · 개발환경', 'score': 0.96405166, 'raw_content': None}, {'url': 'https://velog.io/@qlgks1/python-python%EC%9D%84-%EB%8D%94-pythonic-%ED%95%98%EA%B2%8C-%EC%82%AC%EC%9A%A9%ED%95%98%EA%B8%B

## 5. create_agent와 Tavily Search 통합

이제 Tavily Search를 에이전트와 통합하여 더 지능적인 검색 에이전트를 만들어봅시다.

In [6]:
from langchain.agents import create_agent

# Tavily Search 도구 생성
tavily_tool = TavilySearch(
    max_results=5,
    topic="general",
)

# 검색 에이전트 생성
search_agent = create_agent(
    model="gpt-4o-mini",
    tools=[tavily_tool],
    system_prompt="""당신은 유용한 연구 보조 AI입니다.
    Tavily 검색 도구를 사용하여 정확하고 최신 정보를 찾아주세요.""",
)

In [7]:
for chunk_msg, metadata in search_agent.stream({"messages": [{"role": "user", "content": "2026년의 AI 트렌드는?"}]}, stream_mode="messages"):
    print(chunk_msg.content, end="", flush=True)

{"query": "2026 AI trends", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.ibm.com/think/news/ai-tech-trends-predictions-2026", "title": "The trends that will shape AI and tech in 2026", "content": "2026 will be defined by three trends that move AI beyond personal productivity, says Kevin Chung, Chief Strategy Officer at Writer, an enterprise AI platform for agentic work.\n\n“First, AI is shifting from individual usage to team and workflow orchestration,” Chung told IBM Think. That means coordinating entire workflows, connecting data across departments and moving projects from idea to completion.\n\nSecond, as reasoning capabilities improve, systems won’t just follow instructions: they’ll anticipate needs. “This evolution transforms AI from a passive assistant into an active collaborator capable of meaningful problem-solving and decision-making,” he said. [...] “The most powerful trend I see for next year is AI tackling complex enterprise wo

## 6. 실전 연습 1: 뉴스 분석 에이전트

최신 뉴스를 검색하고 분석하는 에이전트를 만들어봅시다.

In [8]:
# 뉴스 검색용 도구
news_search = TavilySearch(
    max_results=5,
    topic="news",
    include_answer=True,
    start_date="2025-01-01"
)

# 뉴스 분석 에이전트
news_agent = create_agent(
    model="gpt-4o-mini",
    tools=[news_search],
    system_prompt="""당신은 뉴스 분석 보조 AI입니다.
    시사 이슈에 대해 질문을 받으면:
    1. 해당 주제에 대한 최신 뉴스 검색
    2. 핵심 요점 요약
    3. 출처 제공

    분석은 객관적이고 사실에 기반해야 합니다.""",
)

In [9]:
for chunk_msg, metadata in news_agent.stream({"messages": [{"role": "user", "content": "2026년의 밀라노 올림픽 결과"}]}, stream_mode="messages"):
    print(chunk_msg.content, end="", flush=True)

{"query": "2026년 밀라노 올림픽 결과", "follow_up_questions": null, "answer": "The 2026 Milan Winter Olympics saw Alysa Liu of the United States win the gold medal in women's figure skating, with Kaori Sakamoto of Japan taking silver and Ami Nakai of Japan winning bronze. The United States also secured gold in the women's ice hockey, with Mikaela Shiffrin winning gold in the alpine ski, women's slalom race. The U.S. women's team set a record with six golds and 15 medals in women's events.", "images": [], "results": [{"url": "https://apnews.com/photo-gallery/milan-olympics-photos-gallery-day-13-c4139d715f2e359a413173f1af646226", "title": "Olympic photo highlights from Day 13 of the Milan Cortina Winter Games - AP News", "score": 0.7295395, "published_date": "Thu, 19 Feb 2026 16:01:06 GMT", "content": "Test Your News I.Q. 2026 Elections Election Results White House Congress Supreme Court The latest AP-NORC polls Ground Game Early voting tracker. Test Your News I.Q. 2026 Elections Election Results

In [10]:
for chunk_msg, metadata in news_agent.stream({"messages": [{"role": "user", "content": "이란 전쟁으로인한 원유 가격 급등"}]}, stream_mode="messages"):
    print(chunk_msg.content, end="", flush=True)

{"query": "Iran war oil prices surge", "follow_up_questions": null, "answer": "Oil prices have surged significantly due to the ongoing conflict in the Middle East, particularly as Iran's actions have disrupted global oil supplies. Brent futures have risen by more than 50% since the conflict began, briefly topping $119 a barrel. The effective closure of the Strait of Hormuz and attacks on Middle Eastern production facilities have cut deeply into global supplies, leading to Brent crude climbing over 2% in European trade. The West Texas Intermediate futures for May delivery extended gains, rising 3.5% to $106.44 a barrel, marking their biggest monthly surge since 2020. The U.S. national average retail price of gasoline crossed $4 a gallon for the first time in more than three years, driven by the escalating war and geopolitical turmoil.", "images": [], "results": [{"url": "https://www.reuters.com/business/energy/oil-prices-stay-elevated-across-iran-war-scenarios-2026-03-27/", "title": "Oi

## 7. 실전 연습 2: 학술 연구 보조 에이전트

특정 도메인(학술 사이트)만 검색하는 연구 보조 에이전트를 만들어봅시다.

In [11]:
# 학술 검색용 도구
academic_search = TavilySearch(
    max_results=5,
    include_domains=[
        "arxiv.org",
        "scholar.google.com",
        "wikipedia.org",
        "nature.com",
        "sciencedirect.com"
    ],
    search_depth="advanced",
)

# 연구 보조 에이전트
research_agent = create_agent(
    model="gpt-4o-mini",
    tools=[academic_search],
    system_prompt="""당신은 학술 연구 보조 AI입니다.
    사용자가 신뢰할 수 있는 출처에서 학술 정보를 찾도록 도와주세요.
    다음에 중점을 두세요:
    - 과학적 정확성
    - 동료 평가된 출처
    - 최신 연구 결과
    - 복잡한 주제에 대한 명확한 설명

    추가 읽기를 위해 항상 출처 URL을 제공하세요.""",
)

In [12]:
for chunk_msg, metadata in research_agent.stream({"messages": [{"role": "user", "content": "2025년 ~ 2026년의 프롬프트 엔지니어링의 연구 동향을 알려주세요."}]}, stream_mode="messages"):
    print(chunk_msg.content, end="", flush=True)

{"query": "prompt engineering research trends 2025 2026", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://arxiv.org/pdf/2601.01954", "title": "[PDF] Reporting LLM Prompting in Automated Software Engineering - arXiv", "content": "Future work: Our study opens up several directions for future research and community engagement. One promising avenue is to expand the current guidelines toward the emerging paradigms of “promptware”  and “AI-ware”7, where prompts become persistent artifacts embedded in software systems. In these contexts, docu-menting prompts is critical not just for research reproducibility 7 Reporting LLM Prompting in Automated SE: A Guideline Based on Current Practices and Expectations FORGE ’26, April 12–13, 2026, Rio de Janeiro, Brazil but also for long-term maintainability, debugging, and managing technical debt. Alongside prompt engineering, context engineering is gaining importance as a technique for managing information within t

In [13]:
for chunk_msg, metadata in research_agent.stream({"messages": [{"role": "user", "content": "코딩 특화 AI Agent에 대한 연구 자료를 찾아주세요."}]}, stream_mode="messages"):
    print(chunk_msg.content, end="", flush=True)

{"query": "coding specialized AI agent research", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://arxiv.org/html/2603.05344v2", "title": "Building Effective AI Coding Agents for the Terminal - arXiv", "content": "#### 2.2.7 Subagent Orchestration\n\nSome tasks benefit from focused expertise (codebase exploration, user clarification) while others require coordination with main agent state (task management). The main agent can spawn specialized subagents for specific subtasks, each with filtered tool access and specialized prompts. Subagents execute in isolated contexts with their own iteration budgets, preventing unbounded execution.\n\n##### Subagent specialization.\n\nDifferent subagents serve different roles:\n\nCode explorer: Read-only tools for codebase navigation. Specialized for understanding existing code structure.\n\nStrategic planner: Read-only tools plus extended reasoning. Focuses on high-level planning without execution. [...] until 

## 12. 실습 과제

### 📖 과제 1: 도메인 특화 웹 검색 에이전트 만들기

위에서 학습한 내용을 바탕으로, **특정 도메인에 특화된 웹 검색 에이전트**를 만들어보세요.

- 출처를 포함한 정확한 답변 제공

In [15]:
# CODE HERE

<details>
<summary>참고 예시 (클릭하여 펼치기)</summary>

### 레시피 검색 에이전트 예시

```python
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

# 레시피 전문 검색 도구
recipe_search = TavilySearch(
    max_results=5,
    include_domains=[
        "allrecipes.com",
        "foodnetwork.com", 
        "tasty.co",
        "bbcgoodfood.com",
        "10000recipe.com"  # 만개의 레시피
    ],
)

# 레시피 에이전트
recipe_agent = create_agent(
    model="gpt-4o-mini",
    tools=[recipe_search],
    system_prompt="""당신은 친절한 요리 도우미 AI입니다.
    사용자가 요리 레시피를 요청하면:
    1. 신뢰할 수 있는 레시피 사이트에서 최신 정보 검색
    2. 재료, 조리 시간, 난이도를 포함하여 요약
    3. 초보자도 이해하기 쉽게 설명
    4. 출처 URL을 반드시 제공하여 자세한 레시피 확인 가능하도록 함
    
    항상 맛있고 건강한 요리를 만들 수 있도록 도와주세요!""",
)

# 테스트
for chunk_msg, metadata in recipe_agent.stream(
    {"messages": [{"role": "user", "content": "초보자를 위한 간단한 파스타 레시피 알려줘"}]},
    stream_mode="messages"
):
    print(chunk_msg.content, end="", flush=True)
```

</details>

---
### 참고 자료

- [Tavily Search Integration](https://docs.langchain.com/oss/python/integrations/tools/tavily_search)
- [Tavily API Documentation](https://docs.tavily.com/documentation/api-reference/endpoint/search)
- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)